In [ ]:
import os
import glob
import torch
import pandas as pd
from PIL import Image
import numpy as np
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

from transformers import ViTFeatureExtractor, ViTModel
from patches import get_image_patches

mps_device = torch.device("mps")

feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")
model = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k").to(mps_device)
model.eval()

def get_embedding(patch_generator):

    for patch in patch_generator:
        uuid = patch[0]
        info = patch[1]
        images = patch[2]
        inputs = feature_extractor(images=images, return_tensors="pt").to(mps_device)

        with torch.no_grad():
            outputs = model(**inputs)
        X = outputs.last_hidden_state.mean(axis=1).cpu().numpy()

        yield uuid, info, X

In [ ]:
patch_size = 224
window_shift = 112

source_pattern = os.path.join("..", "data", "query", "*fff1.jpg")
images = []
patch_description = []

patch_generator = get_image_patches(
    source_pattern, 
    patch_size, 
    window_shift, 
    batch_size=1024,
    skip=None,
    zoom=0.5
)

embedding_generator = get_embedding(patch_generator)

for uuid, info, patches in patch_generator:
    pass

patch_mean = (patches > 0).mean(axis=1).mean(axis=1).mean(axis=1)
patches = patches[[patch_mean.argmax()], ...]
plt.imshow(patches[0])

In [ ]:
E = list(get_embedding(zip(uuid, info, patches)))

In [ ]:
with h5py.File("../data/vectors.h5", "r") as f:
    vectors = f["vectors"][...]
    uuid = f["uuid"][...]
    info = f["info"][...]

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(E[0][2], vectors, dense_output=True)
sim = pd.DataFrame(
    zip(uuid, info, sim[0]), columns=["uuid", "bbox", "similarity"]
).sort_values(by="similarity", ascending=False).reset_index(drop=True)

In [ ]:
output_dir = os.path.join("..", "data", "artwork")

f, ax = plt.subplots(ncols=4, nrows=8, figsize=(20, 40))
sim_select = sim.drop_duplicates(subset="uuid", keep="first")[:32]
for i, (_, row) in enumerate(sim_select.iterrows()):
    uuid = row["uuid"].decode()
    bbox = row["bbox"]
    file = os.path.join(output_dir, f"{uuid}.jpg")
    image = Image.open(file)
    
    col = i // 4
    row = i % 4
    s = Image.open(file)
    s = np.array(s)

    rect = Rectangle((bbox[2], bbox[0]), bbox[3] - bbox[2], bbox[1] - bbox[0], linewidth=2, edgecolor="r", facecolor="none")

    ax[col, row].imshow(s)
    ax[col, row].axes.get_xaxis().set_ticks([])
    ax[col, row].axes.get_yaxis().set_ticks([])

    ax[col, row].add_patch(rect)